# Approximating SPR Distance Between Phylogenetic Trees with Graph Neural Networks
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2607.18311](https://arxiv.org/abs/2607.18311) (Castanheira, Bugalho, Vaz)
Generated: 2026-07-22

This notebook walks through the key components of the `spr_gnn` implementation,
runs a small-scale training loop on synthetic trees, and shows how to compare
against the paper's reported results (Table 2). It requires no internet access
and no GPU (CPU fallback throughout).

In [ ]:
# Environment check
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU -- training will be slow but the demo below is tiny.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Install the project in editable mode (run once). Safe to re-run.
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])


## Paper overview

Comparing phylogenetic tree topologies is central to tracking epidemics, but the
biologically meaningful **Subtree Prune and Regraft (SPR) distance** is NP-hard to
compute exactly. This paper trains a **Siamese Graph Isomorphism Network (GIN)**
to approximate SPR distance in a single forward pass, using the polynomial-time
`phangorn::SPR.dist` heuristic as a (validated, if biased) supervision target.

Key claims (from the SIR's `provenance.key_claims`):
- A reproducible pre-processing pipeline (slicing, shuffling, UPGMA/NJ inference, midpoint re-rooting, SPR labelling).
- A public dataset of 864 trees / 388 labelled pairs across 4 bacterial species.
- Validation that the phangorn heuristic correlates ~0.98-0.99 (Pearson) with the exact rooted `rspr` distance on small trees, despite underestimating its magnitude.
- A Siamese GIN achieving R² ≈ 0.87-0.90 in-distribution, partial cross-species transfer (R² ≈ 0.37), and failed size extrapolation (R² < 0).

Implementation maps to paper sections as:
| Repo module | Paper section |
|---|---|
| `data/newick_parser.py`, `data/node_features.py` | Sec 4.1, Fig. 4 |
| `models/embedding.py`, `models/gin_encoder.py`, `models/siamese_gin.py` | Sec 4.2, Eq. 1-2, Fig. 5 |
| `training/losses.py`, `training/trainer.py` | Sec 4.3 |
| `evaluation/metrics.py` | Sec 4.3, Sec 5.3, Fig. 7-8 |

## Component walkthrough: Newick parsing

Sec 4.1: *"Each Newick tree is parsed with Biopython and converted to a directed
graph whose root is the unique in-degree-zero vertex... encoded as a
bidirectional graph in PyTorch Geometric so that messages propagate in both
directions."*

In [ ]:
import sys
sys.path.insert(0, "../src")
from spr_gnn.data.newick_parser import NewickToGraph

newick_str = "((A:1,B:1):1,(C:1,D:1):1);"
parser = NewickToGraph()
graph = parser.parse(newick_str)
edge_index = parser.to_bidirectional_edge_index(graph)

print(f"Parsed graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} directed edges")
print(f"Bidirectional edge_index shape: {tuple(edge_index.shape)}  (expected [2, {2*graph.number_of_edges()}])")


## Component walkthrough: node features

Sec 4.1: node degree, binary leaf indicator, topological distance to root
(3 continuous features), plus a categorical taxonomic id later embedded to 16-d
-- concatenated to a 19-d per-node feature vector.

In [ ]:
from spr_gnn.data.node_features import NodeFeatureExtractor

extractor = NodeFeatureExtractor()
continuous, node_ids = extractor.extract(graph, species_id=0)
print(f"Continuous features shape: {tuple(continuous.shape)}  (expected [N,3])")
print(f"Node ids shape:            {tuple(node_ids.shape)}  (expected [N])")
print(continuous)


## Component walkthrough: GIN encoder

Eq. 1 (Sec 4.2): $h_i^{(k)} = \mathrm{MLP}^{(k)}\left((1+\epsilon^{(k)})h_i^{(k-1)} + \sum_{j\in\mathcal{N}(i)}h_j^{(k-1)}\right)$

Eq. 2: $v_g = \sum_{i\in V_g} h_i$ (global add pooling)

**Note (SIR confidence 0.6):** the paper's text says 2 GIN layers; Figure 5's
diagram shows 3. `num_gin_layers` defaults to 2 here and is a one-line config change.

In [ ]:
from spr_gnn.models.gin_encoder import GINEncoder
import torch

encoder = GINEncoder(in_dim=19, hidden_dim=128, num_layers=2)
n_params = sum(p.numel() for p in encoder.parameters())
print(f"GINEncoder parameters: {n_params:,}")

# Toy forward pass: a batch of one graph. Node count must match the actual
# parsed graph (7 nodes for our example tree), derived from edge_index
# rather than hardcoded -- fixes a bug where this was hardcoded to 4.
num_nodes = edge_index.max().item() + 1

x = torch.randn(num_nodes, 19)
batch = torch.zeros(num_nodes, dtype=torch.long)  # all nodes belong to graph 0

out = encoder(x, edge_index, batch)

print(f"Input shape:  {tuple(x.shape)}")
print(f"Output shape: {tuple(out.shape)}  (expected [1, 128])")


## Component walkthrough: full Siamese model

Fig. 5: two trees pass through the *same* GINEncoder (shared weights), their
128-d embeddings are concatenated to 256-d, and an MLP head
(256→128→64→1, dropout 0.3) regresses the SPR distance, clamped at zero.

In [ ]:
from spr_gnn.models.siamese_gin import SiameseGINRegressor
from torch_geometric.data import Data, Batch

model = SiameseGINRegressor(num_species=4, num_gin_layers=2)
n_params = sum(p.numel() for p in model.parameters())
print(f"SiameseGINRegressor parameters: {n_params:,}")

tree_a = Batch.from_data_list([Data(x=continuous, edge_index=edge_index, node_id=node_ids)])
tree_b = Batch.from_data_list([Data(x=continuous, edge_index=edge_index, node_id=node_ids)])
pred = model(tree_a, tree_b)
print(f"Predicted SPR distance for identical tree vs. itself: {pred.item():.4f} (should be small/near zero once trained)")


## Mini-training demonstration (synthetic data, no downloads)

We generate a handful of small random Newick trees in-memory and run a short
training loop end-to-end, to confirm the whole pipeline (data → model → loss →
optimizer) is wired correctly before pointing it at the real Zenodo dataset.

In [ ]:
import random
import pandas as pd
from spr_gnn.data.dataset import TreePairDataset, collate_tree_pairs
from torch.utils.data import DataLoader

def random_newick(leaves=("A","B","C","D","E")):
    leaves = list(leaves)
    random.shuffle(leaves)
    # simple random binary bracketing
    while len(leaves) > 1:
        a, b = leaves.pop(), leaves.pop()
        leaves.append(f"({a}:1,{b}:1)")
    return leaves[0] + ";"

random.seed(0)
import os
os.makedirs("/tmp/spr_gnn_demo_trees", exist_ok=True)
rows = []
for i in range(20):
    t_a, t_b = random_newick(), random_newick()
    path_a, path_b = f"/tmp/spr_gnn_demo_trees/a_{i}.nwk", f"/tmp/spr_gnn_demo_trees/b_{i}.nwk"
    open(path_a, "w").write(t_a)
    open(path_b, "w").write(t_b)
    rows.append({"tree_a_path": path_a, "tree_b_path": path_b, "species": "demo", "spr_label": random.uniform(0, 5)})

df = pd.DataFrame(rows)
species_to_id = {"demo": 0}
dataset = TreePairDataset(df, species_to_id)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_tree_pairs)
print(f"Synthetic dataset: {len(dataset)} pairs, {len(loader)} batches of size 4")


In [ ]:
from spr_gnn.training.losses import HuberRegressionLoss

demo_model = SiameseGINRegressor(num_species=1, num_gin_layers=2).to(device)
optimizer = torch.optim.Adam(demo_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = HuberRegressionLoss(delta=1.0)

demo_model.train()
for step, (tree_a, tree_b, target) in enumerate(loader):
    tree_a, tree_b, target = tree_a.to(device), tree_b.to(device), target.to(device)
    pred = demo_model(tree_a, tree_b)
    loss = criterion(pred, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"step {step}: loss={loss.item():.4f}")


Loss should generally trend downward across these few steps -- this is only a
wiring smoke test on 20 random pairs, not a real training run. For a real run,
download the paper's dataset (`python data/download.py`, see `data/README_data.md`)
and use `python train.py --config configs/config.yaml`.

## Paper's reported results (Table 2, Sec 5.3)

For reference when comparing your own training run's outputs.

In [ ]:
paper_results = {
    "in_distribution":      {"MAE": 127.13, "RMSE": 202.24, "R2": 0.873, "baseline_MAE": 485.78},
    "cross_species":        {"MAE": 208.70, "RMSE": 325.72, "R2": 0.368, "baseline_MAE": 406.05},
    "size_extrapolation":   {"MAE": 375.90, "RMSE": 643.02, "R2": -0.14, "baseline_MAE": 553.65},
    "stratified_cv_overall":{"MAE": "92.24 +/- 7.02", "RMSE": "128.14 +/- 11.07", "MAPE": "2.16% +/- 0.37%", "R2": "0.905 +/- 0.191"},
}
for regime, metrics in paper_results.items():
    print(f"{regime}:")
    for k, v in metrics.items():
        print(f"    {k}: {v}")

print("\nTo reproduce: python train.py --config configs/config.yaml")
print("Then:         python evaluate.py --checkpoint outputs/best_model.pt --regime in_distribution")
print("Cross-species / size-extrapolation regimes need pre-split CSVs -- see data/README_data.md.")


## What to do next

1. **Get the real data**: `python data/download.py` (Zenodo DOI 10.5281/zenodo.20476872), or smoke-test with the bundled `data/toy/` fixture: `python train.py --debug --data-dir data/toy`
2. **Full training**: `python train.py --config configs/config.yaml`
3. **Evaluation**: `python evaluate.py --checkpoint outputs/best_model.pt --regime all`
4. **Compare against the paper**: once you have real results, ArXivist's Stage 6 (Results Comparator) can generate a full comparison report against Table 2.

**Top implementation assumptions to be aware of** (see `sir.json -> implementation_assumptions[]` for the full list):
- `num_gin_layers=2` (SIR confidence 0.6) -- paper text says 2, Figure 5 shows 3. Try both.
- `batch_size=32` (SIR confidence 0.3) -- not stated in the paper at all.
- `huber_delta=1.0` (SIR confidence 0.5) -- not stated in the paper; SPR distances range ~0-2000, may need tuning.